In [5]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

In [6]:
load_dotenv()

True

In [7]:
model = ChatOpenAI(model="gpt-4o-mini")

In [14]:
class SentimentSchema(BaseModel):
    sentiment: Literal["positive", "negative"] = Field(description='Sentiment of the revew')

In [47]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description="The category of issue mentioned in the review")
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description="The emotional tone expresses by the user")
    urgency: Literal["low", "medium", "high"] = Field(desciption="How urgent or critical the issue would be")

C:\Users\nehab\AppData\Local\Temp\ipykernel_18500\3763358411.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'desciption'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  urgency: Literal["low", "medium", "high"] = Field(desciption="How urgent or critical the issue would be")


In [48]:
structured_model = model.with_structured_output(SentimentSchema)
sturctured_model2 = model.with_structured_output(DiagnosisSchema)

In [49]:
prompt = 'What is the sentiment of the folowing review - The software too bad'
structured_model.invoke(prompt).sentiment

'negative'

In [51]:
class ReviewState(TypedDict):
    review: str
    sentiment: Literal["positive", "negative"]
    diagnosis: dict
    response: str

In [52]:
def find_sentiment(state: ReviewState):
    prompt = f'For the following review find out the sentiment \n {state["review"]}'
    sentiment = structured_model.invoke(prompt)
    return {'sentiment': sentiment}

def check_sentiment(state: ReviewState) -> Literal["positive_response", "run_diagnosis"]:

    if state["sentiment"] == 'positive':
        return "positive_response"
    else:
        return "run_diagnosis"
    
def positive_response(state: ReviewState):

    prompt = f"""Write a warm thank you message in response to this review: \n\n\"{state['review']}\"\n. 
    Also, kindly ask the user to leave feedback on our website.
    """
    response = model.invoke(prompt).content
    return {'response': response}

def run_diagnosis(state: ReviewState):

    prompt = f"""Diagnose this negative review: \n\n"{state['review']}\n"\n
    Return issue_type, tone and urgency."""
    response = sturctured_model2.invoke(prompt)

    return {'diagnosis': response.model_dump()}

def negative_response(state: ReviewState):
    diagnosis = state['diagnosis']
    prompt = f""" You are a support assistant. The user has a '{diagnosis['issue_type']}' issue, sounded
    '{diagnosis['tone']} and marked urgency. Write an emphathetic, helpful resolution message.'
"""
    response = model.invoke(prompt).content
    return {'response': response}

In [53]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment', find_sentiment)
graph.add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negative_response', negative_response)

graph.add_edge(START, 'find_sentiment')
graph.add_conditional_edges('find_sentiment', check_sentiment)
graph.add_edge('positive_response', END)
graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)
workflow = graph.compile()


In [55]:
initial_state = {
    'review': 'This app is very disappointing. It crashes often and does not work as expected. The user interface is confusing and not user-friendly at all. Many features are buggy and take too much time to load. Overall experience is poor and it feels like a waste of time. Needs a lot of improvement. Would not recommend.'
}

workflow.invoke(initial_state)

{'review': 'This app is very disappointing. It crashes often and does not work as expected. The user interface is confusing and not user-friendly at all. Many features are buggy and take too much time to load. Overall experience is poor and it feels like a waste of time. Needs a lot of improvement. Would not recommend.',
 'sentiment': SentimentSchema(sentiment='negative'),
 'diagnosis': {'issue_type': 'Bug', 'tone': 'disappointed', 'urgency': 'high'},
 'response': "Subject: We're Here to Help with Your Bug Issue\n\nDear [User's Name],\n\nThank you for reaching out and sharing your concerns with us. I’m truly sorry to hear that you're experiencing a bug; I can understand how frustrating that must be, especially when you have urgent tasks at hand.\n\nYour experience is important to us, and I want to assure you that we’re here to help you resolve this issue as quickly as possible. Could you please provide me with a bit more detail about the bug you’re encountering? Any specific error mess